In [ ]:
import os
import glob
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- CONFIGURE THIS ---
input_csv_folder = r"C:\Users\chris\Desktop\University\Thesis\PipelineResults\10_patients_GM_tiles"
scan_data_root  = r"C:\Users\chris\Desktop\University\Thesis\ScanData\10_patients_GM_tiles"
# ----------------------

csv_files = glob.glob(os.path.join(input_csv_folder, "pipeline_run_*.csv"))

dfs = []
for path in csv_files:
    temp_df = pd.read_csv(path)
    temp_df = temp_df.rename(columns={"scan_folder": "scan_name"})
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df):,} total cells from {len(csv_files)} CSV files")
print(f"Unique images: {df['image_name'].nunique():,}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# --- CONFIGURE THIS ---
N_IMAGES = 5          # how many images to visualise
RANDOM_SEED = 42      # set to None for different picks each run
# ----------------------

image_names = df["image_name"].unique()
print(f"{len(image_names):,} unique images available")

rng = np.random.default_rng(RANDOM_SEED)
chosen = rng.choice(image_names, size=min(N_IMAGES, len(image_names)), replace=False)

print(f"\nPicked {len(chosen)} images to inspect:")
for name in chosen:
    n_boxes = len(df[df["image_name"] == name])
    print(f"  {name}  ({n_boxes} boxes)")

In [ ]:
for image_name in chosen:
    grp = df[df["image_name"] == image_name]
    scan_name = grp["scan_name"].iloc[0]

    img_path = os.path.join(scan_data_root, scan_name, image_name + ".jpg")
    if not os.path.exists(img_path):
        print(f"[SKIP] not found: {img_path}")
        continue

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img_rgb)

    for _, row in grp.iterrows():
        x1, y1, x2, y2 = row["xmin"], row["ymin"], row["xmax"], row["ymax"]
        box_w = x2 - x1
        box_h = y2 - y1
        ax.add_patch(plt.Rectangle(
            (x1, y1), box_w, box_h,
            fill=False, edgecolor="lime", linewidth=1
        ))

    ax.set_title(f"{image_name}\n{len(grp)} boxes  |  image {w}x{h}", fontsize=11)
    ax.axis("off")
    plt.tight_layout()
    plt.show()